# Comparación: CT crudo vs CT preprocesado

Este notebook compara un volumen de `output/CT` (crudo, HU) con su versión en `output/preprocessed/*/CT` para verificar si se aplicó windowing + normalización.

In [1]:
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt

plt.rcParams['figure.figsize'] = (16, 5)
plt.rcParams['image.cmap'] = 'gray'

In [ ]:
def find_project_root(start: Path) -> Path:
    for p in [start, *start.parents]:
        if (p / 'output').exists() and (p / 'src').exists():
            return p
    raise FileNotFoundError('No se encontró la raíz del proyecto con carpetas output/ y src/.')

PROJECT_ROOT = find_project_root(Path.cwd())
OUTPUT_DIR = PROJECT_ROOT / 'output'
PREP_ROOT = OUTPUT_DIR / 'preprocessed'

# Puedes forzar manualmente la carpeta de CT crudos si no está en output/CT.
# Ejemplo: RAW_DIR_OVERRIDE = PROJECT_ROOT / 'otro_lugar' / 'CT'
RAW_DIR_OVERRIDE = None

raw_candidates = [
    OUTPUT_DIR / 'CT',
    OUTPUT_DIR / 'raw' / 'CT',
    OUTPUT_DIR / 'raw_CT',
    PROJECT_ROOT / 'CT',
]
if RAW_DIR_OVERRIDE is not None:
    raw_candidates.insert(0, RAW_DIR_OVERRIDE)

RAW_DIR = next((p for p in raw_candidates if p.exists()), raw_candidates[0])

print('PROJECT_ROOT =', PROJECT_ROOT)
print('RAW_DIR      =', RAW_DIR, '(exists=' + str(RAW_DIR.exists()) + ')')
print('PREP_ROOT    =', PREP_ROOT)

PROJECT_ROOT = c:\TFM
RAW_DIR      = c:\TFM\output\CT
PREP_ROOT    = c:\TFM\output\preprocessed


In [ ]:
# Opcional: fija un paciente concreto, por ejemplo 'LIDC-IDRI-0078'
PATIENT_ID = None

splits = ['train', 'val', 'test']
raw_files = sorted(RAW_DIR.glob('*.npy')) if RAW_DIR.exists() else []
raw_ids = {f.stem for f in raw_files}

prep_map = {}  # patient_id -> (split, path)
for s in splits:
    split_dir = PREP_ROOT / s / 'CT'
    if not split_dir.exists():
        continue
    for f in split_dir.glob('*.npy'):
        prep_map[f.stem] = (s, f)

if not prep_map:
    raise RuntimeError(
        'No se encontraron CT preprocesados en output/preprocessed/{train,val,test}/CT.'
    )

if not raw_ids:
    sample_pid = next(iter(prep_map.keys()))
    split_name, prep_path = prep_map[sample_pid]
    msg = [
        f'No se encontraron CT crudos en: {RAW_DIR}',
        'Para comparar RAW vs PREP necesitas generar/ubicar los .npy crudos.',
        'Opciones:',
        '  1) Ejecutar process_all_masks.py para crear output/CT/*.npy',
        '  2) Ajustar RAW_DIR_OVERRIDE en la celda anterior al directorio correcto.',
        '',
        f'De referencia, sí existe PREP para {sample_pid} en split {split_name}:',
        f'  {prep_path}',
    ]
    raise RuntimeError('\n'.join(msg))

common_ids = sorted(raw_ids.intersection(prep_map.keys()))
if not common_ids:
    example_raw = raw_files[0].name if raw_files else 'N/A'
    example_prep = next(iter(prep_map.keys())) + '.npy'
    msg = [
        'No hay pacientes comunes entre RAW y PREP.',
        f'  RAW dir : {RAW_DIR}',
        f'  PREP dir: {PREP_ROOT}',
        f'  Ejemplo RAW : {example_raw}',
        f'  Ejemplo PREP: {example_prep}',
        'Revisa que ambos conjuntos pertenezcan a la misma corrida/dataset.',
    ]
    raise RuntimeError('\n'.join(msg))

if PATIENT_ID is None:
    PATIENT_ID = common_ids[0]

if PATIENT_ID not in prep_map:
    raise ValueError(f'El paciente {PATIENT_ID} no está en preprocessed.')
if PATIENT_ID not in raw_ids:
    raise ValueError(f'El paciente {PATIENT_ID} no está en RAW ({RAW_DIR}).')

split_name, prep_path = prep_map[PATIENT_ID]
raw_path = RAW_DIR / f'{PATIENT_ID}.npy'

print('Paciente seleccionado:', PATIENT_ID)
print('Split preprocessed    :', split_name)
print('Raw path              :', raw_path)
print('Prep path             :', prep_path)

RuntimeError: No hay pacientes comunes entre output/CT y output/preprocessed/*/CT.

: 

In [ ]:
raw = np.load(raw_path).astype(np.float32)
prep = np.load(prep_path).astype(np.float32)

print('Shape raw :', raw.shape)
print('Shape prep:', prep.shape)
if raw.shape != prep.shape:
    raise RuntimeError('Las formas no coinciden, no se pueden comparar voxel a voxel.')

def describe(name, arr):
    print(f'{name:10s} min={arr.min():9.3f}  max={arr.max():9.3f}  mean={arr.mean():9.3f}  std={arr.std():9.3f}')

describe('RAW', raw)
describe('PREP', prep)

prep_in_01 = (prep.min() >= -1e-6) and (prep.max() <= 1 + 1e-6)
print('PREP en [0,1]:', prep_in_01)
print('PREP % voxels == 0:', float((prep == 0.0).mean() * 100))
print('PREP % voxels == 1:', float((prep == 1.0).mean() * 100))

exact_equal = np.array_equal(raw, prep)
print('RAW y PREP idénticos voxel a voxel:', exact_equal)

In [ ]:
# Verificación fuerte contra la fórmula esperada del pipeline
HU_MIN, HU_MAX = -1000.0, 600.0
raw_windowed = np.clip(raw, HU_MIN, HU_MAX)
raw_expected_norm = (raw_windowed - HU_MIN) / (HU_MAX - HU_MIN)

max_abs_diff = float(np.max(np.abs(prep - raw_expected_norm)))
mean_abs_diff = float(np.mean(np.abs(prep - raw_expected_norm)))
is_same_pipeline = np.allclose(prep, raw_expected_norm, atol=1e-6)

print('Comparación PREP vs norm(clip(RAW)) con HU_MIN=-1000, HU_MAX=600')
print('allclose:', is_same_pipeline)
print('max_abs_diff :', max_abs_diff)
print('mean_abs_diff:', mean_abs_diff)

In [ ]:
# Visualización de un corte axial
z = raw.shape[2] // 2
raw_slice = raw[:, :, z]
prep_slice = prep[:, :, z]
expected_slice = raw_expected_norm[:, :, z]
diff_slice = np.abs(prep_slice - expected_slice)

fig, ax = plt.subplots(1, 4, figsize=(22, 5))
ax[0].imshow(raw_slice, vmin=HU_MIN, vmax=HU_MAX)
ax[0].set_title(f'RAW (HU) z={z}')
ax[0].axis('off')

ax[1].imshow(prep_slice, vmin=0, vmax=1)
ax[1].set_title('PREPROCESADO (0..1)')
ax[1].axis('off')

ax[2].imshow(expected_slice, vmin=0, vmax=1)
ax[2].set_title('Esperado: norm(clip(raw))')
ax[2].axis('off')

im = ax[3].imshow(diff_slice, cmap='magma')
ax[3].set_title('|PREP - esperado|')
ax[3].axis('off')
fig.colorbar(im, ax=ax[3], fraction=0.046, pad=0.04)

plt.tight_layout()
plt.show()

In [ ]:
# Histogramas para comparar distribuciones
fig, ax = plt.subplots(1, 2, figsize=(14, 4))

ax[0].hist(raw.ravel(), bins=200, color='steelblue', alpha=0.9)
ax[0].set_title('Histograma RAW (HU)')
ax[0].set_xlabel('HU')
ax[0].set_ylabel('Frecuencia')

ax[1].hist(prep.ravel(), bins=200, color='darkorange', alpha=0.9)
ax[1].set_title('Histograma PREPROCESADO')
ax[1].set_xlabel('Valor normalizado')
ax[1].set_ylabel('Frecuencia')

plt.tight_layout()
plt.show()

## Interpretación rápida

- Si `PREP en [0,1] = True` y `allclose = True`, entonces el volumen preprocesado coincide con aplicar windowing + normalización al crudo.
- Si `RAW y PREP idénticos = True`, no hubo transformación (caso inesperado en este pipeline).
- Puedes cambiar `PATIENT_ID` en la celda 4 para revisar otros pacientes.